Irrelevant

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

In [2]:
class BandAttention(tf.keras.layers.Layer):
    def __init__(self, num_bands):
        super(BandAttention, self).__init__()
        self.dense1 = layers.Dense(num_bands // 2, activation='relu')  # reduce dimension
        self.dense2 = layers.Dense(num_bands, activation='sigmoid')   # get attention weights

    def call(self, x):
        # x shape: (batch_size, H, W, num_bands)
        band_features = tf.reduce_mean(x, axis=[1, 2])  # Global Average Pooling over spatial dims (H, W)
        attention_weights = self.dense1(band_features)
        attention_weights = self.dense2(attention_weights)  # (batch_size, num_bands)
        attention_weights = tf.expand_dims(tf.expand_dims(attention_weights, 1), 1)  # (batch_size, 1, 1, num_bands)
        return x * attention_weights  # broadcast weights across H and W

In [ ]:
class MultiScaleCNN(tf.keras.layers.Layer):
    def __init__(self):
        super(MultiScaleCNN, self).__init__()
        self.conv3 = layers.Conv2D(64, kernel_size=3, padding='same', activation='relu')
        self.conv5 = layers.Conv2D(64, kernel_size=5, padding='same', activation='relu')
        self.conv7 = layers.Conv2D(64, kernel_size=7, padding='same', activation='relu')
        self.fusion = layers.Conv2D(128, kernel_size=1, activation='relu')

    def call(self, x):
        # Input shape: (batch_size, H, W, num_bands)
        x3 = self.conv3(x)  # fine-grained features (small receptive field)
        x5 = self.conv5(x)  # medium-scale features
        x7 = self.conv7(x)  # large-scale features (captures bigger spatial patterns)
        x_concat = tf.concat([x3, x5, x7], axis=-1)  # concatenate along channels
        return self.fusion(x_concat)  # reduce to 128 channels

In [5]:
class ParallelAttentionFusion(tf.keras.layers.Layer):
    def __init__(self):
        super(ParallelAttentionFusion, self).__init__()
        # fuse spectral and spatial features
        self.fusion = layers.Conv2D(128, kernel_size=3, padding='same', activation='relu')
        self.global_pool = layers.GlobalAveragePooling2D()

    def call(self, spectral_feat, spatial_feat):
        # both inputs: (batch_size, H, W, channels)
        x = tf.concat([spectral_feat, spatial_feat], axis=-1)  # concat spectral and spatial along channel dim
        x = self.fusion(x)  # merge into unified feature map
        return self.global_pool(x)  # reduce spatial dims

In [6]:
class SpectralSpatialAttentionModel(tf.keras.Model):
    def __init__(self, num_bands, num_classes):
        super(SpectralSpatialAttentionModel, self).__init__()
        self.band_attention = BandAttention(num_bands)
        self.spatial_extractor = MultiScaleCNN(in_channels=num_bands)
        self.parallel_fusion = ParallelAttentionFusion()
        self.classifier = layers.Dense(num_classes)  # final classification layer

    def call(self, x):
        # x shape: (batch_size, H, W, num_bands) - TensorFlow uses channels_last
        spectral_feat = self.band_attention(x)  # band-wise attention applied
        spatial_feat = self.spatial_extractor(x)  # multi-scale CNN features
        fused_feat = self.parallel_fusion(spectral_feat, spatial_feat)  # combine both feature types
        return self.classifier(fused_feat)  # logits for classification

In [ ]:
num_bands = 20
num_classes = 4

model = SpectralSpatialAttentionModel(num_bands=num_bands, num_classes=num_classes)
dummy_input = tf.random.normal([8, 224, 224, num_bands])  # batch of 8 images
output = model(dummy_input)
print(output.shape)  # (8, 4)